1. Imports

In [1]:
import ast
import spacy
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity
import csv
import datetime
from datetime import datetime
from transformers import BertTokenizer, BertModel
import torch
import torch.nn.functional as F
from sklearn.cluster import KMeans
import re

c:\Users\korey\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2. Narrative Preprocessing

The Narrative Preprocessing Pipeline is as follows: 

- Import HippoCorpus dataset.
- Process each narrative, such that it is stored as an array, separating each individual sentence.
- Append the narrative story type categorization and the associated summary statement to the story sentence array.
- Export the processed stories.

In [ ]:
hippoCorpusData = pd.read_csv("./hippocorpus-u20220112\hcV3-stories.csv")

In [ ]:
imaginedBaseline = []
for row in hippoCorpusData.iterrows():
    if row[1]['memType'] == 'imagined':
        imaginedBaseline.append(0)
    elif row[1]['memType'] == 'recalled':
        imaginedBaseline.append(1)
    else:
        imaginedBaseline.append(2)

In [ ]:
nlp = spacy.load('en_core_web_sm')

In [ ]:
workableHippoCampusStoryList = []
sentences = []
tempSentence = ""
i = 0

for row in hippoCorpusData.iterrows():
    story = nlp(row[1]['story'])
    for word in story:
        #rint(word)
        tempSentence += str(word) + " "
        if word.lemma_ == '.':
            tempSentence.strip(" ")
            #tempSentence += str(word)
            sentences.append(tempSentence)
            tempSentence = ""
            #Define sentence boundary.
    workableHippoCampusStoryList.append([sentences.copy(), row[1]['memType'], row[1]['summary']])

    if i % 100 == 0:
      print("{} Stories Processed".format(i))

    i = i + 1
    sentences = []

In [ ]:
maxStoryLength = -1
tempStoryLength = -1

for story in workableHippoCampusStoryList:
  tempStoryLength = 0
  for sentence in story[0]:
    print(sentence)
    tempStoryLength += 1


  if tempStoryLength > maxStoryLength:
    maxStoryLength = tempStoryLength

In [ ]:
with open('./hippoCorpusOutput.csv', 'w', newline='', encoding='utf-8') as file:
  writer = csv.writer(file)
  for story in workableHippoCampusStoryList:
    storyHolder = story[0].copy()
    storyHolder.append(story[1])
    storyHolder.append(story[2])
    writer.writerow(storyHolder)

In [ ]:
maxExpectedCols = 40

colNames = [f'col_{i}' for i in range(maxExpectedCols)]
hippoCorpusProcessedDf = pd.read_csv('./hippoCorpusOutput.csv', names=colNames, engine='python')

3. BERT Similarity-by-Sentence (Approach 1) 

In [2]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3986.03it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
storyCount = 0
stabilityArr = []
stabilityComp = 0
stabilityDict = {}
for row in hippoCorpusProcessedDf.iterrows():

  i = 0
  compareFlag = True
  #print(row)
  #print(row[1]['col_{}'.format(i)])

  while type(row[1]['col_{}'.format(i)]) != float:
    #print(row[1]['col_{}'.format(i)])
   
    if row[1]['col_{}'.format(i + 1)] == 'recalled':
      #print("recalled!")
      compareFlag = False
      #stabilityArr.append("recalled")
      if row[1]['col_{}'.format(i + 2)] not in stabilityDict.keys():
        stabilityDict[row[1]['col_{}'.format(i + 2)]] = [{'recalled':stabilityArr.copy()}]
      else:
        stabilityDict[row[1]['col_{}'.format(i + 2)]].append({'recalled':stabilityArr.copy()})
      stabilityArr = []

    elif row[1]['col_{}'.format(i + 1)] == 'imagined':
      #print("imagined!")
      compareFlag = False
      #stabilityArr.append("imagined")
      if row[1]['col_{}'.format(i + 2)] not in stabilityDict.keys():
        stabilityDict[row[1]['col_{}'.format(i + 2)]] = [{'imagined':stabilityArr.copy()}]
      else:
        stabilityDict[row[1]['col_{}'.format(i + 2)]].append({'imagined':stabilityArr.copy()})
      stabilityArr = []
    
    elif row[1]['col_{}'.format(i + 1)] == 'retold':
      compareFlag = False
      #stabilityArr.append("imagined")
      if row[1]['col_{}'.format(i + 2)] not in stabilityDict.keys():
        stabilityDict[row[1]['col_{}'.format(i + 2)]] = [{'retold':stabilityArr.copy()}]
      else:
        stabilityDict[row[1]['col_{}'.format(i + 2)]].append({'retold':stabilityArr.copy()})
      stabilityArr = []

    if compareFlag:
      #print("recalled")
      tokensOne = tokenizer.tokenize(row[1]['col_{}'.format(i)])

      tokensTwo = tokenizer.tokenize(row[1]['col_{}'.format(i + 1)])

      inputIdsOne = torch.tensor(tokenizer.convert_tokens_to_ids(tokensOne)).unsqueeze(0)

      inputIdsTwo = torch.tensor(tokenizer.convert_tokens_to_ids(tokensTwo)).unsqueeze(0)

      with torch.no_grad():

        outputsOne = model(inputIdsOne)

        outputsTwo = model(inputIdsTwo)

        embeddingsOne = F.normalize(outputsOne.last_hidden_state[:, 0, :], p = 2, dim = 1)

        embeddingsTwo = F.normalize(outputsTwo.last_hidden_state[:, 0, :], p = 2, dim = 1)

      xOne = embeddingsOne[0].reshape(1, -1)

      xTwo = embeddingsTwo[0].reshape(1, -1)

      bertCosSims = util.pytorch_cos_sim(xOne, xTwo)
      #print(bertCosSims)
      #print(stabilityComp, row[1]['col_{}'.format(i)], row[1]['col_{}'.format(i + 1)])
      stabilityArr.append(bertCosSims)

  #print(storyCount)
    i = i + 1
  storyCount += 1
  #print(storyCount)
  
  if storyCount % 50 == 0:
    print("{} Stories Counted | {}".format(storyCount, datetime.now()))

In [ ]:
storyCount = 0

with open('./hippoCorpusStabilityBertOutput.csv', 'w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(['story','memStatus','cosSims', 'meanCosSims','sdCosSims'])
    for story in stabilityDict.keys():
    #print(stabilityDict[story])
    #print(len(stabilityDict[story]))

        if storyCount % 50 == 0:
            print("{} Stories Counted | {}".format(storyCount, datetime.now()))

        for i in range(len(stabilityDict[story])):
            #print(story) #Story label/summary
            #print(list(stabilityDict[story][i].keys())[0]) #Story recall/imagined/retold type
            #print(stabilityDict[story][i][list(stabilityDict[story][i].keys())[0]]) Story sentence-level transitional cosine similarities
            #print(np.mean(stabilityDict[story][i][list(stabilityDict[story][i].keys())[0]])) #Mean cosine similarities
            #print(np.std(stabilityDict[story][i][list(stabilityDict[story][i].keys())[0]])) #Standard deviation of cosine similarities

            writer.writerow([story, list(stabilityDict[story][i].keys())[0], stabilityDict[story][i] ,np.mean(stabilityDict[story][i][list(stabilityDict[story][i].keys())[0]]), np.std(stabilityDict[story][i][list(stabilityDict[story][i].keys())[0]])])

            storyCount += 1

In [12]:
stabilityBertData = pd.read_csv('./hippoCorpusSpreadsheets/hippoCorpusStabilityBertOutput.csv')
stabilityBertData.head(5)

,story,memStatus,cosSims,meanCosSims,sdCosSims
0,My boyfriend and I went to a concert together ...,imagined,"{'imagined': [tensor([[0.6826]]), tensor([[0.8...",0.624234,0.107645
1,My boyfriend and I went to a concert together ...,recalled,"{'recalled': [tensor([[0.5659]]), tensor([[0.6...",0.587141,0.077258
2,My sister gave birth to my twin niece and neph...,imagined,"{'imagined': [tensor([[0.7312]]), tensor([[0.7...",0.643608,0.079388
3,My sister gave birth to my twin niece and neph...,recalled,"{'recalled': [tensor([[0.5412]]), tensor([[0.7...",0.561855,0.136896
4,It is always a journey for me to go to burning...,imagined,"{'imagined': [tensor([[0.5770]]), tensor([[0.5...",0.592238,0.080694


3.1. [Mean as Metric]

In [4]:
storyDict = {}
for row in stabilityBertData.iterrows():
    if row[1]['story'] not in storyDict:
        storyDict[row[1]['story']] = [{row[1]['memStatus']:row[1]['meanCosSims']}]
    else:
        storyDict[row[1]['story']].append({row[1]['memStatus']:row[1]['meanCosSims']})

storyTypeList = []
storyImaginedArr = []
storyRecalledArr = []
imaginedMoreStableCount = 0
recalledMoreStableCount = 0

firstPassImagined = True
firstPassRecalled = True
for entry in storyDict.keys():
    #print(storyDict[entry])
    for i in range(len(storyDict[entry])):
        storyTypeList.append(list(storyDict[entry][i].keys())[0])
        #print(storyTypeList[i])
        #print(list(storyTypeList)[0])
  
        if 'imagined' in storyTypeList[i] and firstPassImagined:
            storyImaginedArr.append(list(storyDict[entry][i].values())[0])
            firstPassImagined = False
        elif 'recalled' in storyTypeList[i] and firstPassRecalled:
            storyRecalledArr.append(list(storyDict[entry][i].values())[0])
            firstPassRecalled = False
        #elif 'retold' in storyTypeList[i]:
        #    storyRecalledArr.append(list(storyDict[entry][i].values())[0])
    
        #print(storyImaginedArr, storyRecalledArr)
    

    if 'imagined' in storyTypeList and 'recalled' in storyTypeList:
        if np.mean(storyImaginedArr) < np.mean(storyRecalledArr):
            imaginedMoreStableCount += 1
        else:
            recalledMoreStableCount += 1
        #print(storyImaginedArr, storyRecalledArr)
        #print(imaginedMoreStableCount, recalledMoreStableCount)
    #break
    firstPassImagined = True
    firstPassRecalled = True
    storyTypeList = []
    storyImaginedArr = []
    storyRecalledArr = []

In [5]:
print(imaginedMoreStableCount, recalledMoreStableCount)

1278 1294


3.2. [Standard Deviation as Metric]

In [26]:
storyDict = {}
for row in stabilityBertData.iterrows():
    if row[1]['story'] not in storyDict:
        storyDict[row[1]['story']] = [{row[1]['memStatus']:row[1]['sdCosSims']}]
    else:
        storyDict[row[1]['story']].append({row[1]['memStatus']:row[1]['sdCosSims']})

storyTypeList = []
storyImaginedArr = []
storyRecalledArr = []
imaginedMoreStableCount = 0
recalledMoreStableCount = 0

firstPassImagined = True
firstPassRecalled = True
for entry in storyDict.keys():
    #print(storyDict[entry])
    for i in range(len(storyDict[entry])):
        storyTypeList.append(list(storyDict[entry][i].keys())[0])
        #print(storyTypeList[i])
        #print(list(storyTypeList)[0])
  
        if 'imagined' in storyTypeList[i] and firstPassImagined:
            storyImaginedArr.append(list(storyDict[entry][i].values())[0])
            firstPassImagined = False
        elif 'recalled' in storyTypeList[i] and firstPassRecalled:
            storyRecalledArr.append(list(storyDict[entry][i].values())[0])
            firstPassRecalled = False
        #elif 'retold' in storyTypeList[i]:
        #    storyRecalledArr.append(list(storyDict[entry][i].values())[0])
    
        #print(storyImaginedArr, storyRecalledArr)
    

    if 'imagined' in storyTypeList and 'recalled' in storyTypeList:
        if np.mean(storyImaginedArr) < np.mean(storyRecalledArr):
            imaginedMoreStableCount += 1
        else:
            recalledMoreStableCount += 1
        #print(storyImaginedArr, storyRecalledArr)
        #print(imaginedMoreStableCount, recalledMoreStableCount)
    #break
    firstPassImagined = True
    firstPassRecalled = True
    storyTypeList = []
    storyImaginedArr = []
    storyRecalledArr = []

In [27]:
print(imaginedMoreStableCount, recalledMoreStableCount)

1315 1257


4. S-BERT Similarity-by-Sentence (Approach 1)

In [20]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11196.99it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


- Implementation analagous to Section 3.

In [11]:
stabilitySbertData = pd.read_csv('./hippoCorpusSpreadsheets/hippoCorpusStabilitySbertOutput.csv')
stabilitySbertData.head(5)

,story,memStatus,cosSims,meanCosSims,sdCosSims
0,My boyfriend and I went to a concert together ...,imagined,"{'imagined': [np.float32(0.6281102), np.float3...",0.336175,0.128445
1,My boyfriend and I went to a concert together ...,recalled,"{'recalled': [np.float32(0.3056181), np.float3...",0.294264,0.100239
2,My sister gave birth to my twin niece and neph...,imagined,"{'imagined': [np.float32(0.22942325), np.float...",0.321054,0.142022
3,My sister gave birth to my twin niece and neph...,recalled,"{'recalled': [np.float32(0.66162765), np.float...",0.400626,0.148018
4,It is always a journey for me to go to burning...,imagined,"{'imagined': [np.float32(0.19643974), np.float...",0.219971,0.097393


4.1. [Mean as Metric]

In [10]:
storyDict = {}
for row in stabilitySbertData.iterrows():
    if row[1]['story'] not in storyDict:
        storyDict[row[1]['story']] = [{row[1]['memStatus']:row[1]['meanCosSims']}]
    else:
        storyDict[row[1]['story']].append({row[1]['memStatus']:row[1]['meanCosSims']})

In [11]:
storyTypeList = []
storyImaginedArr = []
storyRecalledArr = []
imaginedMoreStableCount = 0
recalledMoreStableCount = 0

firstPassImagined = True
firstPassRecalled = True
for entry in storyDict.keys():
    #print(storyDict[entry])
    for i in range(len(storyDict[entry])):
        storyTypeList.append(list(storyDict[entry][i].keys())[0])
        #print(storyTypeList[i])
        #print(list(storyTypeList)[0])
  
        if 'imagined' in storyTypeList[i] and firstPassImagined:
            storyImaginedArr.append(list(storyDict[entry][i].values())[0])
            firstPassImagined = False
        elif 'recalled' in storyTypeList[i] and firstPassRecalled:
            storyRecalledArr.append(list(storyDict[entry][i].values())[0])
            firstPassRecalled = False
        #elif 'retold' in storyTypeList[i]:
        #    storyRecalledArr.append(list(storyDict[entry][i].values())[0])
    
        #print(storyImaginedArr, storyRecalledArr)
    

    if 'imagined' in storyTypeList and 'recalled' in storyTypeList:
        if np.mean(storyImaginedArr) < np.mean(storyRecalledArr):
            imaginedMoreStableCount += 1
        else:
            recalledMoreStableCount += 1
        #print(storyImaginedArr, storyRecalledArr)
        #print(imaginedMoreStableCount, recalledMoreStableCount)
    #break
    firstPassImagined = True
    firstPassRecalled = True
    storyTypeList = []
    storyImaginedArr = []
    storyRecalledArr = []

In [12]:
print(imaginedMoreStableCount, recalledMoreStableCount)

1490 1082


4.2. [Standard Deviation as Metric]

In [28]:
storyDict = {}
for row in stabilitySbertData.iterrows():
    if row[1]['story'] not in storyDict:
        storyDict[row[1]['story']] = [{row[1]['memStatus']:row[1]['sdCosSims']}]
    else:
        storyDict[row[1]['story']].append({row[1]['memStatus']:row[1]['sdCosSims']})

In [29]:
storyTypeList = []
storyImaginedArr = []
storyRecalledArr = []
imaginedMoreStableCount = 0
recalledMoreStableCount = 0

firstPassImagined = True
firstPassRecalled = True
for entry in storyDict.keys():
    #print(storyDict[entry])
    for i in range(len(storyDict[entry])):
        storyTypeList.append(list(storyDict[entry][i].keys())[0])
        #print(storyTypeList[i])
        #print(list(storyTypeList)[0])
  
        if 'imagined' in storyTypeList[i] and firstPassImagined:
            storyImaginedArr.append(list(storyDict[entry][i].values())[0])
            firstPassImagined = False
        elif 'recalled' in storyTypeList[i] and firstPassRecalled:
            storyRecalledArr.append(list(storyDict[entry][i].values())[0])
            firstPassRecalled = False
        #elif 'retold' in storyTypeList[i]:
        #    storyRecalledArr.append(list(storyDict[entry][i].values())[0])
    
        #print(storyImaginedArr, storyRecalledArr)
    

    if 'imagined' in storyTypeList and 'recalled' in storyTypeList:
        if np.mean(storyImaginedArr) < np.mean(storyRecalledArr):
            imaginedMoreStableCount += 1
        else:
            recalledMoreStableCount += 1
        #print(storyImaginedArr, storyRecalledArr)
        #print(imaginedMoreStableCount, recalledMoreStableCount)
    #break
    firstPassImagined = True
    firstPassRecalled = True
    storyTypeList = []
    storyImaginedArr = []
    storyRecalledArr = []

In [30]:
print(imaginedMoreStableCount, recalledMoreStableCount)

1385 1187


5. S3BERT Similarity-by-Sentence (Approach 1)

In [ ]:
maxExpectedCols = 40

colNames = [f'col_{i}' for i in range(maxExpectedCols)]
hippoCorpusProcessedDf = pd.read_csv('/content/content/hippoCorpusOutput.csv', names=colNames, engine='python')

In [ ]:
storyCount = 0
stabilityArr = []
stabilityComp = 0
stabilityDict = {}
for row in hippoCorpusProcessedDf.iterrows():

  i = 0
  compareFlag = True
  #print(row)
  #print(row[1]['col_{}'.format(i)])

  while type(row[1]['col_{}'.format(i)]) != float:
    #print(row[1]['col_{}'.format(i)])

    if row[1]['col_{}'.format(i + 1)] == 'recalled':
      #print("recalled!")
      compareFlag = False
      #stabilityArr.append("recalled")
      if row[1]['col_{}'.format(i + 2)] not in stabilityDict.keys():
        stabilityDict[row[1]['col_{}'.format(i + 2)]] = [{'recalled':stabilityArr.copy()}]
      else:
        stabilityDict[row[1]['col_{}'.format(i + 2)]].append({'recalled':stabilityArr.copy()})
      stabilityArr = []

    elif row[1]['col_{}'.format(i + 1)] == 'imagined':
      #print("imagined!")
      compareFlag = False
      #stabilityArr.append("imagined")
      if row[1]['col_{}'.format(i + 2)] not in stabilityDict.keys():
        stabilityDict[row[1]['col_{}'.format(i + 2)]] = [{'imagined':stabilityArr.copy()}]
      else:
        stabilityDict[row[1]['col_{}'.format(i + 2)]].append({'imagined':stabilityArr.copy()})
      stabilityArr = []

    elif row[1]['col_{}'.format(i + 1)] == 'retold':
      compareFlag = False
      #stabilityArr.append("imagined")
      if row[1]['col_{}'.format(i + 2)] not in stabilityDict.keys():
        stabilityDict[row[1]['col_{}'.format(i + 2)]] = [{'retold':stabilityArr.copy()}]
      else:
        stabilityDict[row[1]['col_{}'.format(i + 2)]].append({'retold':stabilityArr.copy()})
      stabilityArr = []

    if compareFlag:
      #print("recalled")
      #reload(s3bert_infer)
      sentenceCosSims = get_features(row[1]['col_{}'.format(i)], row[1]['col_{}'.format(i + 1)])

      #print(stabilityComp, row[1]['col_{}'.format(i)], row[1]['col_{}'.format(i + 1)])
      stabilityArr.append(sentenceCosSims)

  #print(storyCount)
    i = i + 1
  storyCount += 1
  print(storyCount)
  if storyCount % 50 == 0:
    print("{} Stories Counted | {}".format(storyCount, datetime.now()))



In [ ]:
path = '/content/drive/MyDrive/MyStuff/Programming/s3bertHippoCampusStabilityResultsUpToFinished.csv'
df = pd.read_csv(path)

5.1. [Mean as Metric]

In [ ]:
firstPassImagined = True
firstPassRecalled = True

imaginedStoriesStable = 0
recalledStoriesStable = 0

storyCount = 0

patternO = r"^\{'(imagined|recalled)':"



for label, values in df.items():

  averageImaginedCosSims = 0
  averageRecalledCosSims = 0

  #print(label)
  i = 0
  #while type(values[i]) == str and i < 50:
  while i < len(values) and not pd.isna(values[i]):
    match = re.search(patternO, values[i])
    if match:
      category = match.group(1)
    #print(f"Category identified: {category}")
    #print(values[i])


    pattern = r"'global'\s*:\s*np\.float64\((.*?)\)"

    global_values = re.findall(pattern, str(values[i]))

    global_floats = [float(val) for val in global_values]
    i = i + 1

    if category == 'imagined':
      firstPassImagined = False
      averageImaginedCosSims = np.mean(global_floats)
      #print(averageImaginedCosSims

    if category == 'recalled':
      firstPassRecalled = False
      averageRecalledCosSims = np.mean(global_floats)

  if firstPassImagined == False and firstPassRecalled == False:
    if averageImaginedCosSims < averageRecalledCosSims:
      imaginedStoriesStable = imaginedStoriesStable + 1
    else:
      recalledStoriesStable = recalledStoriesStable + 1
  firstPassImagined = True
  firstPassRecalled = True
  storyCount += 1

  if storyCount % 50 == 0:
    print("{} Stories Counted | {}".format(storyCount, datetime.now()))
    print(imaginedStoriesStable, recalledStoriesStable)
  #print(firstPassImagined, firstPassRecalled, recalledStoriesStable, imaginedStoriesStable)

In [ ]:
#2750 Stories Counted | 2026-07-14 22:39:47.799431
#print(imaginedStoriesStable, recalledStoriesStable)
#1347 1225

5.2. [Standard Deviation as Metric]

In [ ]:
firstPassImagined = True
firstPassRecalled = True

imaginedStoriesStable = 0
recalledStoriesStable = 0

storyCount = 0

patternO = r"^\{'(imagined|recalled)':"

for label, values in df.items():

  averageImaginedCosSims = 0
  averageRecalledCosSims = 0

  #print(label)
  i = 0
  #while type(values[i]) == str and i < 50:
  while i < len(values) and not pd.isna(values[i]):
    match = re.search(patternO, values[i])
    if match:
      category = match.group(1)
    #print(f"Category identified: {category}")
    #print(values[i])

    pattern = r"'global'\s*:\s*np\.float64\((.*?)\)"

    global_values = re.findall(pattern, str(values[i]))

    global_floats = [float(val) for val in global_values]
    i = i + 1

    if category == 'imagined':
      firstPassImagined = False
      averageImaginedCosSims = np.std(global_floats)
      #print(averageImaginedCosSims

    if category == 'recalled':
      firstPassRecalled = False
      averageRecalledCosSims = np.std(global_floats)

  if firstPassImagined == False and firstPassRecalled == False:
    if averageImaginedCosSims < averageRecalledCosSims:
      imaginedStoriesStable = imaginedStoriesStable + 1
    else:
      recalledStoriesStable = recalledStoriesStable + 1
  firstPassImagined = True
  firstPassRecalled = True
  storyCount += 1

  if storyCount % 50 == 0:
    print("{} Stories Counted | {}".format(storyCount, datetime.now()))
    print(imaginedStoriesStable, recalledStoriesStable)
  #print(firstPassImagined, firstPassRecalled, recalledStoriesStable, imaginedStoriesStable)

In [ ]:
#2750 Stories Counted | 2026-07-14 22:37:19.806915
#print(imaginedStoriesStable, recalledStoriesStable)
#1316 1256

6. BERT Similarity-to-Summary (Approach 2) 

In [13]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9649.99it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
storyCount = 0
stabilityArr = []
stabilityComp = 0
stabilityDict = {}
for row in hippoCorpusProcessedDf.iterrows():

  i = 0
  j = 0
  compareFlag = True
  #print(row)
  #print(row[1]['col_{}'.format(i)])

  while type(row[1]['col_{}'.format(j)]) != float: #Getting the summary statement 
    j = j + 1
  j = j - 1
  summaryStatement = row[1]['col_{}'.format(j)]

  while type(row[1]['col_{}'.format(i)]) != float:
    #print(row[1]['col_{}'.format(i)])
   
    if row[1]['col_{}'.format(i + 1)] == 'recalled':
      #print("recalled!")
      compareFlag = False
      #stabilityArr.append("recalled")
      if row[1]['col_{}'.format(i + 2)] not in stabilityDict.keys():
        stabilityDict[row[1]['col_{}'.format(i + 2)]] = [{'recalled':stabilityArr.copy()}]
      else:
        stabilityDict[row[1]['col_{}'.format(i + 2)]].append({'recalled':stabilityArr.copy()})
      stabilityArr = []

    elif row[1]['col_{}'.format(i + 1)] == 'imagined':
      #print("imagined!")
      compareFlag = False
      #stabilityArr.append("imagined")
      if row[1]['col_{}'.format(i + 2)] not in stabilityDict.keys():
        stabilityDict[row[1]['col_{}'.format(i + 2)]] = [{'imagined':stabilityArr.copy()}]
      else:
        stabilityDict[row[1]['col_{}'.format(i + 2)]].append({'imagined':stabilityArr.copy()})
      stabilityArr = []
    
    elif row[1]['col_{}'.format(i + 1)] == 'retold':
      compareFlag = False
      #stabilityArr.append("imagined")
      if row[1]['col_{}'.format(i + 2)] not in stabilityDict.keys():
        stabilityDict[row[1]['col_{}'.format(i + 2)]] = [{'retold':stabilityArr.copy()}]
      else:
        stabilityDict[row[1]['col_{}'.format(i + 2)]].append({'retold':stabilityArr.copy()})
      stabilityArr = []

    if compareFlag:
      #print("recalled")
      tokensOne = tokenizer.tokenize(row[1]['col_{}'.format(i)])

      tokensTwo = tokenizer.tokenize(summaryStatement)

      inputIdsOne = torch.tensor(tokenizer.convert_tokens_to_ids(tokensOne)).unsqueeze(0)

      inputIdsTwo = torch.tensor(tokenizer.convert_tokens_to_ids(tokensTwo)).unsqueeze(0)

      with torch.no_grad():

        outputsOne = model(inputIdsOne)

        outputsTwo = model(inputIdsTwo)

        embeddingsOne = F.normalize(outputsOne.last_hidden_state[:, 0, :], p = 2, dim = 1)

        embeddingsTwo = F.normalize(outputsTwo.last_hidden_state[:, 0, :], p = 2, dim = 1)

      xOne = embeddingsOne[0].reshape(1, -1)

      xTwo = embeddingsTwo[0].reshape(1, -1)

      bertCosSims = util.pytorch_cos_sim(xOne, xTwo)
      #print(bertCosSims)
      #print(stabilityComp, row[1]['col_{}'.format(i)], row[1]['col_{}'.format(i + 1)])
      stabilityArr.append(bertCosSims)

  #print(storyCount)
    i = i + 1
  storyCount += 1
  #print(storyCount)
  
  if storyCount % 50 == 0:
    print("{} Stories Counted | {}".format(storyCount, datetime.now()))

In [ ]:
storyCount = 0

with open('./hippoCorpusStabilityBertTopicOutput.csv', 'w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(['story','memStatus','cosSims', 'meanCosSims','sdCosSims'])
    for story in stabilityDict.keys():
    #print(stabilityDict[story])
    #print(len(stabilityDict[story]))

        if storyCount % 50 == 0:
            print("{} Stories Counted | {}".format(storyCount, datetime.now()))

        for i in range(len(stabilityDict[story])):
            #print(story) #Story label/summary
            #print(list(stabilityDict[story][i].keys())[0]) #Story recall/imagined/retold type
            #print(stabilityDict[story][i][list(stabilityDict[story][i].keys())[0]]) Story sentence-level transitional cosine similarities
            #print(np.mean(stabilityDict[story][i][list(stabilityDict[story][i].keys())[0]])) #Mean cosine similarities
            #print(np.std(stabilityDict[story][i][list(stabilityDict[story][i].keys())[0]])) #Standard deviation of cosine similarities

            writer.writerow([story, list(stabilityDict[story][i].keys())[0], stabilityDict[story][i] ,np.mean(stabilityDict[story][i][list(stabilityDict[story][i].keys())[0]]), np.std(stabilityDict[story][i][list(stabilityDict[story][i].keys())[0]])])

            storyCount += 1

In [7]:
stabilityBertTopicData = pd.read_csv('./hippoCorpusSpreadsheets/hippoCorpusStabilityBertTopicOutput.csv')
stabilityBertTopicData.head(5)

,story,memStatus,cosSims,meanCosSims,sdCosSims
0,My boyfriend and I went to a concert together ...,imagined,"{'imagined': [tensor([[0.6953]]), tensor([[0.8...",0.682029,0.125400
1,My boyfriend and I went to a concert together ...,recalled,"{'recalled': [tensor([[0.7754]]), tensor([[0.6...",0.629321,0.113829
2,My sister gave birth to my twin niece and neph...,imagined,"{'imagined': [tensor([[0.7351]]), tensor([[0.6...",0.632040,0.087521
3,My sister gave birth to my twin niece and neph...,recalled,"{'recalled': [tensor([[0.5336]]), tensor([[0.7...",0.555764,0.171425
4,It is always a journey for me to go to burning...,imagined,"{'imagined': [tensor([[0.4833]]), tensor([[0.6...",0.521608,0.169489


6.1. [Mean as Metric]

In [16]:
storyDict = {}
for row in stabilityBertTopicData.iterrows():
    if row[1]['story'] not in storyDict:
        storyDict[row[1]['story']] = [{row[1]['memStatus']:row[1]['meanCosSims']}]
    else:
        storyDict[row[1]['story']].append({row[1]['memStatus']:row[1]['meanCosSims']})

In [17]:
storyTypeList = []
storyImaginedArr = []
storyRecalledArr = []
imaginedMoreStableCount = 0
recalledMoreStableCount = 0

firstPassImagined = True
firstPassRecalled = True
for entry in storyDict.keys():
    #print(storyDict[entry])
    for i in range(len(storyDict[entry])):
        storyTypeList.append(list(storyDict[entry][i].keys())[0])
        #print(storyTypeList[i])
        #print(list(storyTypeList)[0])
  
        if 'imagined' in storyTypeList[i] and firstPassImagined:
            storyImaginedArr.append(list(storyDict[entry][i].values())[0])
            firstPassImagined = False
        elif 'recalled' in storyTypeList[i] and firstPassRecalled:
            storyRecalledArr.append(list(storyDict[entry][i].values())[0])
            firstPassRecalled = False
        #elif 'retold' in storyTypeList[i]:
        #    storyRecalledArr.append(list(storyDict[entry][i].values())[0])
    
        #print(storyImaginedArr, storyRecalledArr)
    

    if 'imagined' in storyTypeList and 'recalled' in storyTypeList:
        if np.mean(storyImaginedArr) < np.mean(storyRecalledArr):
            imaginedMoreStableCount += 1
        else:
            recalledMoreStableCount += 1
        #print(storyImaginedArr, storyRecalledArr)
        #print(imaginedMoreStableCount, recalledMoreStableCount)
    #break
    firstPassImagined = True
    firstPassRecalled = True
    storyTypeList = []
    storyImaginedArr = []
    storyRecalledArr = []

In [ ]:
print(imaginedMoreStableCount, recalledMoreStableCount) 

1569 1003


6.2. [Standard Deviation as Metric]

In [31]:
storyDict = {}
for row in stabilityBertTopicData.iterrows():
    if row[1]['story'] not in storyDict:
        storyDict[row[1]['story']] = [{row[1]['memStatus']:row[1]['sdCosSims']}]
    else:
        storyDict[row[1]['story']].append({row[1]['memStatus']:row[1]['sdCosSims']})

In [32]:
storyTypeList = []
storyImaginedArr = []
storyRecalledArr = []
imaginedMoreStableCount = 0
recalledMoreStableCount = 0

firstPassImagined = True
firstPassRecalled = True
for entry in storyDict.keys():
    #print(storyDict[entry])
    for i in range(len(storyDict[entry])):
        storyTypeList.append(list(storyDict[entry][i].keys())[0])
        #print(storyTypeList[i])
        #print(list(storyTypeList)[0])
  
        if 'imagined' in storyTypeList[i] and firstPassImagined:
            storyImaginedArr.append(list(storyDict[entry][i].values())[0])
            firstPassImagined = False
        elif 'recalled' in storyTypeList[i] and firstPassRecalled:
            storyRecalledArr.append(list(storyDict[entry][i].values())[0])
            firstPassRecalled = False
        #elif 'retold' in storyTypeList[i]:
        #    storyRecalledArr.append(list(storyDict[entry][i].values())[0])
    
        #print(storyImaginedArr, storyRecalledArr)
    

    if 'imagined' in storyTypeList and 'recalled' in storyTypeList:
        if np.mean(storyImaginedArr) < np.mean(storyRecalledArr):
            imaginedMoreStableCount += 1
        else:
            recalledMoreStableCount += 1
        #print(storyImaginedArr, storyRecalledArr)
        #print(imaginedMoreStableCount, recalledMoreStableCount)
    #break
    firstPassImagined = True
    firstPassRecalled = True
    storyTypeList = []
    storyImaginedArr = []
    storyRecalledArr = []

In [ ]:
print(imaginedMoreStableCount, recalledMoreStableCount) 

1310 1262


7. S-BERT Similarity-to-Summary (Approach 2)

In [19]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11174.11it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


- Implementation analagous to Section 6.

In [6]:
stabilitySbertTopicData = pd.read_csv('./hippoCorpusSpreadsheets/hippoCorpusStabilitySbertTopicOutput.csv')
stabilitySbertTopicData.head(5)

,story,memStatus,cosSims,meanCosSims,sdCosSims
0,My boyfriend and I went to a concert together ...,imagined,"{'imagined': [np.float32(0.56309825), np.float...",0.394372,0.126074
1,My boyfriend and I went to a concert together ...,recalled,"{'recalled': [np.float32(0.41139832), np.float...",0.354763,0.128180
2,My sister gave birth to my twin niece and neph...,imagined,"{'imagined': [np.float32(0.018267341), np.floa...",0.245043,0.157067
3,My sister gave birth to my twin niece and neph...,recalled,"{'recalled': [np.float32(0.618824), np.float32...",0.398737,0.175445
4,It is always a journey for me to go to burning...,imagined,"{'imagined': [np.float32(0.38085192), np.float...",0.310165,0.162698


7.1. [Mean as Metric]

In [23]:
storyDict = {}
for row in stabilitySbertTopicData.iterrows():
    if row[1]['story'] not in storyDict:
        storyDict[row[1]['story']] = [{row[1]['memStatus']:row[1]['meanCosSims']}]
    else:
        storyDict[row[1]['story']].append({row[1]['memStatus']:row[1]['meanCosSims']})

In [24]:
storyTypeList = []
storyImaginedArr = []
storyRecalledArr = []
imaginedMoreStableCount = 0
recalledMoreStableCount = 0

firstPassImagined = True
firstPassRecalled = True
for entry in storyDict.keys():
    #print(storyDict[entry])
    for i in range(len(storyDict[entry])):
        storyTypeList.append(list(storyDict[entry][i].keys())[0])
        #print(storyTypeList[i])
        #print(list(storyTypeList)[0])
  
        if 'imagined' in storyTypeList[i] and firstPassImagined:
            storyImaginedArr.append(list(storyDict[entry][i].values())[0])
            firstPassImagined = False
        elif 'recalled' in storyTypeList[i] and firstPassRecalled:
            storyRecalledArr.append(list(storyDict[entry][i].values())[0])
            firstPassRecalled = False
        elif 'retold' in storyTypeList[i]:
            storyRecalledArr.append(list(storyDict[entry][i].values())[0])
    
        #print(storyImaginedArr, storyRecalledArr)
    

    if 'imagined' in storyTypeList and 'recalled' in storyTypeList:
        if np.mean(storyImaginedArr) < np.mean(storyRecalledArr):
            imaginedMoreStableCount += 1
        else:
            recalledMoreStableCount += 1
        #print(storyImaginedArr, storyRecalledArr)
        #print(imaginedMoreStableCount, recalledMoreStableCount)
    #break
    firstPassImagined = True
    firstPassRecalled = True
    storyTypeList = []
    storyImaginedArr = []
    storyRecalledArr = []

In [25]:
print(imaginedMoreStableCount, recalledMoreStableCount)

1700 872


7.2. [Standard Deviation as Metric]

In [34]:
storyDict = {}
for row in stabilitySbertTopicData.iterrows():
    if row[1]['story'] not in storyDict:
        storyDict[row[1]['story']] = [{row[1]['memStatus']:row[1]['sdCosSims']}]
    else:
        storyDict[row[1]['story']].append({row[1]['memStatus']:row[1]['sdCosSims']})

In [35]:
storyTypeList = []
storyImaginedArr = []
storyRecalledArr = []
imaginedMoreStableCount = 0
recalledMoreStableCount = 0

firstPassImagined = True
firstPassRecalled = True
for entry in storyDict.keys():
    #print(storyDict[entry])
    for i in range(len(storyDict[entry])):
        storyTypeList.append(list(storyDict[entry][i].keys())[0])
        #print(storyTypeList[i])
        #print(list(storyTypeList)[0])
  
        if 'imagined' in storyTypeList[i] and firstPassImagined:
            storyImaginedArr.append(list(storyDict[entry][i].values())[0])
            firstPassImagined = False
        elif 'recalled' in storyTypeList[i] and firstPassRecalled:
            storyRecalledArr.append(list(storyDict[entry][i].values())[0])
            firstPassRecalled = False
        elif 'retold' in storyTypeList[i]:
            storyRecalledArr.append(list(storyDict[entry][i].values())[0])
    
        #print(storyImaginedArr, storyRecalledArr)
    

    if 'imagined' in storyTypeList and 'recalled' in storyTypeList:
        if np.mean(storyImaginedArr) < np.mean(storyRecalledArr):
            imaginedMoreStableCount += 1
        else:
            recalledMoreStableCount += 1
        #print(storyImaginedArr, storyRecalledArr)
        #print(imaginedMoreStableCount, recalledMoreStableCount)
    #break
    firstPassImagined = True
    firstPassRecalled = True
    storyTypeList = []
    storyImaginedArr = []
    storyRecalledArr = []

In [36]:
print(imaginedMoreStableCount, recalledMoreStableCount)

1325 1247


8. S3BERT Similarity-to-Summary (Approach 2)

In [ ]:
def get_features(sentA, sentB): #Implementation in S3BERT framework
# example sentence pairs
  xsent = [sentA]
  ysent = [sentB]

# encode with s3bert
  xsent_encoded = model.encode(xsent, normalize_embeddings=True)
  ysent_encoded = model.encode(ysent, normalize_embeddings=True)

# get similarity scores of different features
  preds = ph.get_preds(xsent_encoded, ysent_encoded, biases=None, n=config.N, dim=config.FEATURE_DIM) #Note, that config references the 'config.py' file retrieved from the S3BERT pipeline.

# print similarity scores of different features
  features = ["global"] + config.FEATURES[2:] + ["residual"]
  for i, x in enumerate(xsent):
    sims = preds[i]
    jl = {k:v for k,v in zip(features, sims)}
    jl["sent_a"] = x
    jl["sent_b"] = ysent[i]
    #print(jl)

  return jl


In [ ]:
storyCount = 0
stabilityArr = []
stabilityComp = 0
stabilityDict = {}
for row in hippoCorpusProcessedDf.iterrows():

  i = 0
  j = 0
  k = 0
  compareFlag = True
  #print(row)
  #print(row[1]['col_{}'.format(i)])

  while type(row[1]['col_{}'.format(j)]) != float: #Getting the summary statement
    j = j + 1
    if row[1]['col_{}'.format(j)] == 'recalled' or row[1]['col_{}'.format(j)] == 'imagined' or row[1]['col_{}'.format(j)] == 'retold':
      k = j

      #print(j)
  summaryStatement = row[1]['col_{}'.format(k + 1)]
  #print(summaryStatement)
  #break

  while type(row[1]['col_{}'.format(i)]) != float:
    #print(row[1]['col_{}'.format(i)])

    if row[1]['col_{}'.format(i + 1)] == 'recalled':
      #print("recalled!")
      compareFlag = False
      #stabilityArr.append("recalled")
      if row[1]['col_{}'.format(i + 2)] not in stabilityDict.keys():
        stabilityDict[row[1]['col_{}'.format(i + 2)]] = [{'recalled':stabilityArr.copy()}]
      else:
        stabilityDict[row[1]['col_{}'.format(i + 2)]].append({'recalled':stabilityArr.copy()})
      stabilityArr = []

    elif row[1]['col_{}'.format(i + 1)] == 'imagined':
      #print("imagined!")
      compareFlag = False
      #stabilityArr.append("imagined")
      if row[1]['col_{}'.format(i + 2)] not in stabilityDict.keys():
        stabilityDict[row[1]['col_{}'.format(i + 2)]] = [{'imagined':stabilityArr.copy()}]
      else:
        stabilityDict[row[1]['col_{}'.format(i + 2)]].append({'imagined':stabilityArr.copy()})
      stabilityArr = []

    elif row[1]['col_{}'.format(i + 1)] == 'retold':
      compareFlag = False
      #stabilityArr.append("imagined")
      if row[1]['col_{}'.format(i + 2)] not in stabilityDict.keys():
        stabilityDict[row[1]['col_{}'.format(i + 2)]] = [{'retold':stabilityArr.copy()}]
      else:
        stabilityDict[row[1]['col_{}'.format(i + 2)]].append({'retold':stabilityArr.copy()})
      stabilityArr = []

    if compareFlag:
      #print("recalled")
      #reload(s3bert_infer)
      sentenceCosSims = get_features(row[1]['col_{}'.format(i)], summaryStatement)

      #print(stabilityComp, row[1]['col_{}'.format(i)], row[1]['col_{}'.format(i + 1)])
      stabilityArr.append(sentenceCosSims)

  #print(storyCount)
    i = i + 1
  storyCount += 1
  print(storyCount)
  if storyCount % 50 == 0:
    print("{} Stories Counted | {}".format(storyCount, datetime.now()))

  if storyCount % 1000 == 0:
    df = pd.DataFrame({k: pd.Series(v) for k, v in stabilityDict.items()})

    path = '/content/drive/MyDrive/MyStuff/Programming/s3bertHippoCampusStabilityResultsVersionTwoUpToFinished.csv'
    df.to_csv(path, index=False)

In [ ]:
path = '/content/drive/MyDrive/MyStuff/Programming/s3bertHippoCampusStabilityResultsVersionTwoUpToFinished.csv'
df = pd.read_csv(path)

8.1. [Mean as Metric]

In [ ]:
firstPassImagined = True
firstPassRecalled = True

imaginedStoriesStable = 0
recalledStoriesStable = 0

storyCount = 0

patternO = r"^\{'(imagined|recalled)':"

for label, values in df.items():

  averageImaginedCosSims = 0
  averageRecalledCosSims = 0

  #print(label)
  i = 0
  #while type(values[i]) == str and i < 50:
  while i < len(values) and not pd.isna(values[i]):
    match = re.search(patternO, values[i])
    if match:
      category = match.group(1)
    #print(f"Category identified: {category}")
    #print(values[i])


    pattern = r"'global'\s*:\s*np\.float64\((.*?)\)"

    global_values = re.findall(pattern, str(values[i]))

    global_floats = [float(val) for val in global_values]
    i = i + 1

    if category == 'imagined':
      firstPassImagined = False
      averageImaginedCosSims = np.mean(global_floats)
      #print(averageImaginedCosSims

    if category == 'recalled':
      firstPassRecalled = False
      averageRecalledCosSims = np.mean(global_floats)

  if firstPassImagined == False and firstPassRecalled == False:
    if averageImaginedCosSims < averageRecalledCosSims:
      imaginedStoriesStable = imaginedStoriesStable + 1
    else:
      recalledStoriesStable = recalledStoriesStable + 1
  firstPassImagined = True
  firstPassRecalled = True
  storyCount += 1

  if storyCount % 50 == 0:
    print("{} Stories Counted | {}".format(storyCount, datetime.now()))
    print(imaginedStoriesStable, recalledStoriesStable)
  #print(firstPassImagined, firstPassRecalled, recalledStoriesStable, imaginedStoriesStable)

In [ ]:
#2750 Stories Counted | 2026-07-14 20:31:53.652995 (Some stories are skipped as they are not the first instance of a recalled or imagined story)
#print(imaginedStoriesStable, recalledStoriesStable)
#1618 954

8.2. [Standard Deviation as Metric]

In [ ]:
firstPassImagined = True
firstPassRecalled = True

imaginedStoriesStable = 0
recalledStoriesStable = 0

storyCount = 0

patternO = r"^\{'(imagined|recalled)':"



for label, values in df.items():

  averageImaginedCosSims = 0
  averageRecalledCosSims = 0

  #print(label)
  i = 0
  #while type(values[i]) == str and i < 50:
  while i < len(values) and not pd.isna(values[i]):
    match = re.search(patternO, values[i])
    if match:
      category = match.group(1)
    #print(f"Category identified: {category}")
    #print(values[i])


    pattern = r"'global'\s*:\s*np\.float64\((.*?)\)"

    global_values = re.findall(pattern, str(values[i]))

    global_floats = [float(val) for val in global_values]
    i = i + 1

    if category == 'imagined':
      firstPassImagined = False
      averageImaginedCosSims = np.std(global_floats)
      #print(averageImaginedCosSims

    if category == 'recalled':
      firstPassRecalled = False
      averageRecalledCosSims = np.std(global_floats)

  if firstPassImagined == False and firstPassRecalled == False:
    if averageImaginedCosSims < averageRecalledCosSims:
      imaginedStoriesStable = imaginedStoriesStable + 1
    else:
      recalledStoriesStable = recalledStoriesStable + 1
  firstPassImagined = True
  firstPassRecalled = True
  storyCount += 1

  if storyCount % 50 == 0:
    print("{} Stories Counted | {}".format(storyCount, datetime.now()))
    print(imaginedStoriesStable, recalledStoriesStable)
  #print(firstPassImagined, firstPassRecalled, recalledStoriesStable, imaginedStoriesStable)

In [ ]:
#2750 Stories Counted | 2026-07-14 22:31:48.634408 (Some stories are skipped as they are not the first instance of a recalled or imagined story)
#print(imaginedStoriesStable, recalledStoriesStable)
#1387 1185

9. BERT Distance-to-Summary (Approach 3)

9.1. Model Imports

In [3]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4762.56it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


9.2. Distance-to-Summary Analyses

In [ ]:
storyCount = 0
vecArr = []
stabilityComp = 0
stabilityDict = {}


for row in hippoCorpusProcessedDf.iterrows():

  i = 0
  compareFlag = True
  #print(row)
  #print(row[1]['col_{}'.format(i)])

  while type(row[1]['col_{}'.format(i)]) != float:
    #print(row[1]['col_{}'.format(i)])

    if row[1]['col_{}'.format(i)] == 'recalled':
      #print("recalled!")
      compareFlag = False
      #stabilityArr.append("recalled")
      if row[1]['col_{}'.format(i + 1)] not in stabilityDict.keys():

        kmeans = KMeans(n_clusters=1, init='k-means++', random_state=42)
        kmeans.fit_predict(np.array(vecArr))

        tokensTopicSentence = tokenizer.tokenize(row[1]['col_{}'.format(i + 1)])

        inputIdsTopic = torch.tensor(tokenizer.convert_tokens_to_ids(tokensTopicSentence)).unsqueeze(0) 

        with torch.no_grad():

            outputsTopicSentence = model(inputIdsTopic)

        topicSentenceEmbeds = F.normalize(outputsTopicSentence.last_hidden_state[:, 0, :], p = 2, dim = 1)[0] #Extract recalled summary sentences.
        
        centroids = kmeans.cluster_centers_
        #print(len(centroids[0]))
        #print(len(topicSentenceEmbeds))

        euclidDistance = np.linalg.norm(torch.from_numpy(centroids[0]) - topicSentenceEmbeds) #Calculates Euclidean distances from narrative cluster sentence centroids to topic sentence embedding.
        #print("Euclidean Distance: {}".format(euclidDistance))

        stabilityDict[row[1]['col_{}'.format(i + 1)]] = [{'recalled':euclidDistance}] 
        vecArr = []

      else:

        kmeans = KMeans(n_clusters=1, init='k-means++', random_state=42)
        kmeans.fit_predict(np.array(vecArr))

        tokensTopicSentence = tokenizer.tokenize(row[1]['col_{}'.format(i + 1)])

        inputIdsTopic = torch.tensor(tokenizer.convert_tokens_to_ids(tokensTopicSentence)).unsqueeze(0)

        with torch.no_grad():

            outputsTopicSentence = model(inputIdsTopic)

        topicSentenceEmbeds = F.normalize(outputsTopicSentence.last_hidden_state[:, 0, :], p = 2, dim = 1)[0]
        
        centroids = kmeans.cluster_centers_
        #print(len(centroids[0]))
        #print(len(topicSentenceEmbeds))

        euclidDistance = np.linalg.norm(torch.from_numpy(centroids[0]) - topicSentenceEmbeds)
        #print("Euclidean Distance: {}".format(euclidDistance))

        stabilityDict[row[1]['col_{}'.format(i + 1)]].append({'recalled':euclidDistance})
        vecArr = []

    elif row[1]['col_{}'.format(i)] == 'imagined':
      
      #print("imagined!")
      compareFlag = False
      #stabilityArr.append("imagined")
      if row[1]['col_{}'.format(i + 1)] not in stabilityDict.keys():

        kmeans = KMeans(n_clusters=1, init='k-means++', random_state=42)
        kmeans.fit_predict(np.array(vecArr))

        tokensTopicSentence = tokenizer.tokenize(row[1]['col_{}'.format(i + 1)])

        inputIdsTopic = torch.tensor(tokenizer.convert_tokens_to_ids(tokensTopicSentence)).unsqueeze(0)

        with torch.no_grad():

            outputsTopicSentence = model(inputIdsTopic)

        topicSentenceEmbeds = F.normalize(outputsTopicSentence.last_hidden_state[:, 0, :], p = 2, dim = 1)[0]
        
        centroids = kmeans.cluster_centers_
        #print(len(centroids[0]))
        #print(len(topicSentenceEmbeds))

        euclidDistance = np.linalg.norm(torch.from_numpy(centroids[0]) - topicSentenceEmbeds)
        #print("Euclidean Distance: {}".format(euclidDistance))

        stabilityDict[row[1]['col_{}'.format(i + 1)]] = [{'imagined':euclidDistance}]
        vecArr = []
      else:
        kmeans = KMeans(n_clusters=1, init='k-means++', random_state=42)
        kmeans.fit_predict(np.array(vecArr))

        tokensTopicSentence = tokenizer.tokenize(row[1]['col_{}'.format(i + 1)])

        inputIdsTopic = torch.tensor(tokenizer.convert_tokens_to_ids(tokensTopicSentence)).unsqueeze(0)

        with torch.no_grad():

            outputsTopicSentence = model(inputIdsTopic)

        topicSentenceEmbeds = F.normalize(outputsTopicSentence.last_hidden_state[:, 0, :], p = 2, dim = 1)[0]
        
        centroids = kmeans.cluster_centers_
        #print(len(centroids[0]))
        #print(len(topicSentenceEmbeds))

        euclidDistance = np.linalg.norm(torch.from_numpy(centroids[0]) - topicSentenceEmbeds)
        #print("Euclidean Distance: {}".format(euclidDistance))

        stabilityDict[row[1]['col_{}'.format(i + 1)]].append({'imagined':euclidDistance})
        vecArr = []
    
    elif row[1]['col_{}'.format(i)] == 'retold':
      compareFlag = False
      #stabilityArr.append("imagined")
      if row[1]['col_{}'.format(i + 1)] not in stabilityDict.keys():

        kmeans = KMeans(n_clusters=1, init='k-means++', random_state=42)
        kmeans.fit_predict(np.array(vecArr))

        tokensTopicSentence = tokenizer.tokenize(row[1]['col_{}'.format(i + 1)])

        inputIdsTopic = torch.tensor(tokenizer.convert_tokens_to_ids(tokensTopicSentence)).unsqueeze(0)

        with torch.no_grad():

            outputsTopicSentence = model(inputIdsTopic)

        topicSentenceEmbeds = F.normalize(outputsTopicSentence.last_hidden_state[:, 0, :], p = 2, dim = 1)[0]
        
        centroids = kmeans.cluster_centers_
        #print(len(centroids[0]))
        #print(len(topicSentenceEmbeds))

        euclidDistance = np.linalg.norm(torch.from_numpy(centroids[0]) - topicSentenceEmbeds)
        #print("Euclidean Distance: {}".format(euclidDistance))

        stabilityDict[row[1]['col_{}'.format(i + 1)]] = [{'retold':euclidDistance}]
        vecArr = []
      else:

        kmeans = KMeans(n_clusters=1, init='k-means++', random_state=42)
        kmeans.fit_predict(np.array(vecArr))

        tokensTopicSentence = tokenizer.tokenize(row[1]['col_{}'.format(i + 1)])

        inputIdsTopic = torch.tensor(tokenizer.convert_tokens_to_ids(tokensTopicSentence)).unsqueeze(0)

        with torch.no_grad():

            outputsTopicSentence = model(inputIdsTopic)

        topicSentenceEmbeds = F.normalize(outputsTopicSentence.last_hidden_state[:, 0, :], p = 2, dim = 1)[0]
        
        centroids = kmeans.cluster_centers_
        #print(len(centroids[0]))
        #print(len(topicSentenceEmbeds))

        euclidDistance = np.linalg.norm(torch.from_numpy(centroids[0]) - topicSentenceEmbeds)
        #print("Euclidean Distance: {}".format(euclidDistance))

        stabilityDict[row[1]['col_{}'.format(i + 1)]].append({'retold':euclidDistance})
        vecArr = []

    if compareFlag:
      #print("recalled")

      tokensOne = tokenizer.tokenize(row[1]['col_{}'.format(i)])

      inputIdsOne = torch.tensor(tokenizer.convert_tokens_to_ids(tokensOne)).unsqueeze(0)

      #

      with torch.no_grad():

          outputsOne = model(inputIdsOne)

      sentenceEmbeds = F.normalize(outputsOne.last_hidden_state[:, 0, :], p = 2, dim = 1)[0]

      #print(sentenceEmbeds.shape)    
      vecArr.append(sentenceEmbeds)
      #print(len(vecArr))
      #print(stabilityComp, row[1]['col_{}'.format(i)], row[1]['col_{}'.format(i + 1)])

    i = i + 1
    #print(i)
  storyCount += 1
  #break
  #print(storyCount)
  

  
  if storyCount % 50 == 0:
    print("{} Stories Counted | {}".format(storyCount, datetime.now()))

In [8]:
stabilityBertClusterData = pd.read_csv('./hippoCorpusSpreadsheets/hippoCorpusStabilityBertClusteringVersionTwoOutput.csv')
stabilityBertClusterData.head(5)

,story,memStatus,euclidDistanceTopicCluster
0,My boyfriend and I went to a concert together ...,imagined,{'imagined': np.float32(0.5134625)}
1,My boyfriend and I went to a concert together ...,recalled,{'recalled': np.float32(0.59943056)}
2,My sister gave birth to my twin niece and neph...,imagined,{'imagined': np.float32(0.63504994)}
3,My sister gave birth to my twin niece and neph...,recalled,{'recalled': np.float32(0.6883018)}
4,It is always a journey for me to go to burning...,imagined,{'imagined': np.float32(0.7630834)}


In [ ]:
storyCount = 0

with open('./hippoCorpusStabilityBertClusteringVersionTwoOutput.csv', 'w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(['story','memStatus','euclidDistanceTopicCluster'])
    for story in stabilityDict.keys():
    #print(stabilityDict[story])
    #print(len(stabilityDict[story]))

        if storyCount % 50 == 0:
            print("{} Stories Counted | {}".format(storyCount, datetime.now()))

        for i in range(len(stabilityDict[story])):
            #print(story) #Story label/summary
            #print(list(stabilityDict[story][i].keys())[0]) #Story recall/imagined/retold type
            #print(stabilityDict[story][i][list(stabilityDict[story][i].keys())[0]]) Story sentence-level transitional cosine similarities
            #print(np.mean(stabilityDict[story][i][list(stabilityDict[story][i].keys())[0]])) #Mean cosine similarities
            #print(np.std(stabilityDict[story][i][list(stabilityDict[story][i].keys())[0]])) #Standard deviation of cosine similarities

            writer.writerow([story, list(stabilityDict[story][i].keys())[0], stabilityDict[story][i]])

            storyCount += 1

In [9]:
stabilityBertClusterData = pd.read_csv('./hippoCorpusSpreadsheets/hippoCorpusStabilityBertClusteringVersionTwoOutput.csv')
stabilityBertClusterData.head(5)

,story,memStatus,euclidDistanceTopicCluster
0,My boyfriend and I went to a concert together ...,imagined,{'imagined': np.float32(0.5134625)}
1,My boyfriend and I went to a concert together ...,recalled,{'recalled': np.float32(0.59943056)}
2,My sister gave birth to my twin niece and neph...,imagined,{'imagined': np.float32(0.63504994)}
3,My sister gave birth to my twin niece and neph...,recalled,{'recalled': np.float32(0.6883018)}
4,It is always a journey for me to go to burning...,imagined,{'imagined': np.float32(0.7630834)}


In [58]:
storyDict = {}
for row in stabilityBertClusterData.iterrows():
    if row[1]['story'] not in storyDict:
        storyDict[row[1]['story']] = [{row[1]['memStatus']:row[1]['euclidDistanceTopicCluster']}]
    else:
        storyDict[row[1]['story']].append({row[1]['memStatus']:row[1]['euclidDistanceTopicCluster']})

In [60]:
def cleanNpString(string):
    return re.sub(r'np\.\w+\((.*?)\)', r'\1', string)

In [ ]:
storyTypeList = []
storyImaginedArr = 0
storyRecalledArr = 0
storyRetoldArr = []
imaginedMoreStableCount = 0
recalledMoreStableCount = 0

firstPassImagined = True
firstPassRecalled = True
for entry in storyDict.keys():
    #print(entry)
    if entry in list(stabilityBertClusterData['story']):
        
    #break
    #print(storyDict[entry])
        for i in range(len(storyDict[entry])):
            storyTypeList.append(list(storyDict[entry][i].keys())[0])
        #print(storyTypeList[i])
        #print(list(storyTypeList)[0])
            print(storyDict[entry][i].keys())
            if 'imagined' in storyTypeList[i] and firstPassImagined:
                storyImaginedArr = storyDict[entry][i].values()
                jsonString = cleanNpString(list(storyImaginedArr)[0])
                dataDict = ast.literal_eval(jsonString)

                imaginedEuclidDistance = dataDict['imagined']
                firstPassImagined = False

            elif 'recalled' in storyTypeList[i] and firstPassRecalled:
                
                storyRecalledArr = storyDict[entry][i].values()

                jsonString = cleanNpString(list(storyRecalledArr)[0])
                
                dataDict = ast.literal_eval(jsonString)
                

                recalledEuclidDistance = dataDict['recalled']
                firstPassRecalled = False

     
        #print(storyImaginedArr, storyRecalledArr)
    
        if 'imagined' in storyTypeList and 'recalled' in storyTypeList:
            if imaginedEuclidDistance < recalledEuclidDistance:
                imaginedMoreStableCount = imaginedMoreStableCount + 1
            else:
                recalledMoreStableCount = recalledMoreStableCount + 1
    #break
        #print(storyImaginedArr, storyRecalledArr)
        #print(imaginedMoreStableCount, recalledMoreStableCount)
    #break
    firstPassImagined = True
    firstPassRecalled = True
    storyTypeList = []
    storyImaginedArr = []
    storyRecalledArr = []

In [ ]:
print(imaginedMoreStableCount, recalledMoreStableCount) #Lower kmeans inertia interpreted as greater stability of narrative subtopic shift.

933 1639


10. S-BERT Distance-to-Summary (Approach 3)

In [64]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8702.05it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


- Implementation analagous to Section 9.

In [10]:
stabilitySbertClusterData = pd.read_csv('./hippoCorpusSpreadsheets/hippoCorpusStabilitySbertClusteringVersionTwoOutput.csv')
stabilitySbertClusterData.head(5)

,story,memStatus,euclidDistanceTopicCluster
0,My boyfriend and I went to a concert together ...,imagined,{'imagined': np.float32(0.7587259)}
1,My boyfriend and I went to a concert together ...,recalled,{'recalled': np.float32(0.7882962)}
2,My sister gave birth to my twin niece and neph...,imagined,{'imagined': np.float32(0.8731619)}
3,My sister gave birth to my twin niece and neph...,recalled,{'recalled': np.float32(0.7692952)}
4,It is always a journey for me to go to burning...,imagined,{'imagined': np.float32(0.8161302)}


In [66]:
storyDict = {}
for row in stabilitySbertClusterData.iterrows():
    if row[1]['story'] not in storyDict:
        storyDict[row[1]['story']] = [{row[1]['memStatus']:row[1]['euclidDistanceTopicCluster']}]
    else:
        storyDict[row[1]['story']].append({row[1]['memStatus']:row[1]['euclidDistanceTopicCluster']})

In [67]:
def cleanNpString(string):
    return re.sub(r'np\.\w+\((.*?)\)', r'\1', string)

In [ ]:
storyTypeList = []
storyImaginedArr = 0
storyRecalledArr = 0
storyRetoldArr = []
imaginedMoreStableCount = 0
recalledMoreStableCount = 0

firstPassImagined = True
firstPassRecalled = True
for entry in storyDict.keys():
    #print(entry)
    if entry in list(stabilitySbertClusterData['story']):
        
    #break
    #print(storyDict[entry])
        for i in range(len(storyDict[entry])):
            storyTypeList.append(list(storyDict[entry][i].keys())[0])
        #print(storyTypeList[i])
        #print(list(storyTypeList)[0])
            print(storyDict[entry][i].keys())
            if 'imagined' in storyTypeList[i] and firstPassImagined:
                storyImaginedArr = storyDict[entry][i].values()
                jsonString = cleanNpString(list(storyImaginedArr)[0])
                dataDict = ast.literal_eval(jsonString)

                imaginedEuclidDistance = dataDict['imagined']
                firstPassImagined = False

            elif 'recalled' in storyTypeList[i] and firstPassRecalled:
                
                storyRecalledArr = storyDict[entry][i].values()

                jsonString = cleanNpString(list(storyRecalledArr)[0])
                
                dataDict = ast.literal_eval(jsonString)
                

                recalledEuclidDistance = dataDict['recalled']
                firstPassRecalled = False

     
        #print(storyImaginedArr, storyRecalledArr)
    
        if 'imagined' in storyTypeList and 'recalled' in storyTypeList:
            if imaginedEuclidDistance < recalledEuclidDistance:
                imaginedMoreStableCount = imaginedMoreStableCount + 1
            else:
                recalledMoreStableCount = recalledMoreStableCount + 1
    #break
        #print(storyImaginedArr, storyRecalledArr)
        #print(imaginedMoreStableCount, recalledMoreStableCount)
    #break
    firstPassImagined = True
    firstPassRecalled = True
    storyTypeList = []
    storyImaginedArr = []
    storyRecalledArr = []

In [69]:
print(imaginedMoreStableCount, recalledMoreStableCount) #Lower euclidean distance between topic sentence and cluster centroid is better.

762 1810


11. S3BERT Distance-to-Summary (Approach 3)

In [ ]:
model = SentenceTransformer("./" + config.SBERT_SAVE_PATH + "/", device="cpu") #S3BERT Model

def getEncoded(sentA):

  use_cuda = torch.cuda.is_available()
  device = torch.device("cuda" if use_cuda else "cpu")

  xsent = [sentA]

  xsent_encoded = model.encode(xsent, normalize_embeddings=True)

  return xsent_encoded

In [ ]:
storyCount = 0
vecArr = []
stabilityComp = 0
stabilityDict = {}

for row in hippoCorpusProcessedDf.iterrows():

  i = 0
  compareFlag = True
  #print(row)
  #print(row[1]['col_{}'.format(i)])

  while type(row[1]['col_{}'.format(i)]) != float:
    #print(row[1]['col_{}'.format(i)])

    if row[1]['col_{}'.format(i)] == 'recalled':
      #print("recalled!")
      compareFlag = False
      #stabilityArr.append("recalled")
      if row[1]['col_{}'.format(i + 1)] not in stabilityDict.keys():

        kmeans = KMeans(n_clusters=1, init='k-means++', random_state=42)
        kmeans.fit_predict(np.array(vecArr))

        topicSentenceEmbeds = getEncoded(row[1]['col_{}'.format(i + 1)])

        centroids = kmeans.cluster_centers_
        #print(len(centroids[0]))
        #print(len(topicSentenceEmbeds))

        euclidDistance = np.linalg.norm(centroids[0] - topicSentenceEmbeds)
        #print("Euclidean Distance: {}".format(euclidDistance))

        stabilityDict[row[1]['col_{}'.format(i + 1)]] = [{'recalled':euclidDistance}]
        vecArr = []

      else:

        kmeans = KMeans(n_clusters=1, init='k-means++', random_state=42)
        kmeans.fit_predict(np.array(vecArr))

        topicSentenceEmbeds = getEncoded(row[1]['col_{}'.format(i + 1)])

        centroids = kmeans.cluster_centers_
        #print(len(centroids[0]))
        #print(len(topicSentenceEmbeds))

        euclidDistance = np.linalg.norm(centroids[0] - topicSentenceEmbeds)
        #print("Euclidean Distance: {}".format(euclidDistance))

        stabilityDict[row[1]['col_{}'.format(i + 1)]].append({'recalled':euclidDistance})
        vecArr = []

    elif row[1]['col_{}'.format(i)] == 'imagined':

      #print("imagined!")
      compareFlag = False
      #stabilityArr.append("imagined")
      if row[1]['col_{}'.format(i + 1)] not in stabilityDict.keys():

        kmeans = KMeans(n_clusters=1, init='k-means++', random_state=42)
        kmeans.fit_predict(np.array(vecArr))

        topicSentenceEmbeds = getEncoded(row[1]['col_{}'.format(i + 1)])

        centroids = kmeans.cluster_centers_
        #print(len(centroids[0]))
        #print(len(topicSentenceEmbeds))

        euclidDistance = np.linalg.norm(centroids[0] - topicSentenceEmbeds)
        #print("Euclidean Distance: {}".format(euclidDistance))

        stabilityDict[row[1]['col_{}'.format(i + 1)]] = [{'imagined':euclidDistance}]
        vecArr = []
      else:
        kmeans = KMeans(n_clusters=1, init='k-means++', random_state=42)
        kmeans.fit_predict(np.array(vecArr))

        topicSentenceEmbeds = getEncoded(row[1]['col_{}'.format(i + 1)])

        centroids = kmeans.cluster_centers_
        #print(len(centroids[0]))
        #print(len(topicSentenceEmbeds))

        euclidDistance = np.linalg.norm(centroids[0] - topicSentenceEmbeds)
        #print("Euclidean Distance: {}".format(euclidDistance))

        stabilityDict[row[1]['col_{}'.format(i + 1)]].append({'imagined':euclidDistance})
        vecArr = []

    elif row[1]['col_{}'.format(i)] == 'retold':
      compareFlag = False
      #stabilityArr.append("imagined")
      if row[1]['col_{}'.format(i + 1)] not in stabilityDict.keys():

        kmeans = KMeans(n_clusters=1, init='k-means++', random_state=42)
        kmeans.fit_predict(np.array(vecArr))

        topicSentenceEmbeds = getEncoded(row[1]['col_{}'.format(i + 1)])

        centroids = kmeans.cluster_centers_
        #print(len(centroids[0]))
        #print(len(topicSentenceEmbeds))

        euclidDistance = np.linalg.norm(centroids[0] - topicSentenceEmbeds)
        #print("Euclidean Distance: {}".format(euclidDistance))

        stabilityDict[row[1]['col_{}'.format(i + 1)]] = [{'retold':euclidDistance}]
        vecArr = []
      else:

        kmeans = KMeans(n_clusters=1, init='k-means++', random_state=42)
        kmeans.fit_predict(np.array(vecArr))

        topicSentenceEmbeds = getEncoded(row[1]['col_{}'.format(i + 1)])

        centroids = kmeans.cluster_centers_
        #print(len(centroids[0]))
        #print(len(topicSentenceEmbeds))

        euclidDistance = np.linalg.norm(centroids[0] - topicSentenceEmbeds)
        #print("Euclidean Distance: {}".format(euclidDistance))

        stabilityDict[row[1]['col_{}'.format(i + 1)]].append({'retold':euclidDistance})
        vecArr = []

    if compareFlag:
      #print("recalled")
      sentenceEmbeds = getEncoded(row[1]['col_{}'.format(i)])

      vecArr.append(sentenceEmbeds[0])
      #print(sentenceEmbeds[0].shape)
      #print(len(vecArr))
      #print(stabilityComp, row[1]['col_{}'.format(i)], row[1]['col_{}'.format(i + 1)])

    i = i + 1
    #print(i)
  storyCount += 1
  #break
  #print(storyCount)


  if storyCount % 50 == 0:
    print("{} Stories Counted | {}".format(storyCount, datetime.now()))

In [ ]:
storyCount = 0

with open('/content/drive/MyDrive/MyStuff/Programming/hippoCorpusStabilityS3BertClusteringVersionTwoOutput.csv', 'w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(['story','memStatus','euclidDistance'])
    for story in stabilityDict.keys():
    #print(stabilityDict[story])
    #print(len(stabilityDict[story]))

        if storyCount % 500 == 0:
            print("{} Stories Counted | {}".format(storyCount, datetime.now()))

        for i in range(len(stabilityDict[story])):
            #print(story) #Story label/summary
            #print(list(stabilityDict[story][i].keys())[0]) #Story recall/imagined/retold type
            #print(stabilityDict[story][i][list(stabilityDict[story][i].keys())[0]]) Story sentence-level transitional cosine similarities
            #print(np.mean(stabilityDict[story][i][list(stabilityDict[story][i].keys())[0]])) #Mean cosine similarities
            #print(np.std(stabilityDict[story][i][list(stabilityDict[story][i].keys())[0]])) #Standard deviation of cosine similarities

            writer.writerow([story, list(stabilityDict[story][i].keys())[0], stabilityDict[story][i]])

            storyCount += 1

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/MyStuff/Programming/hippoCorpusStabilityS3BertClusteringVersionTwoOutput.csv")

imaginedStoryLowerDistance = 0
recalledStoryLowerDistance = 0
storyCount = 0

for story_id, story_df in df.groupby('story'):
    
    imaginedStatusCheck = False
    recalledStatusCheck = False
    imaginedEuclidDistance = -1
    recalledEuclidDistance = -1

    for _, row in story_df.iterrows():
       
        if row['memStatus'] == 'imagined' and not imaginedStatusCheck:
            imaginedStatusCheck = True
            imaginedEuclidDistance = eval(row['euclidDistance'])['imagined']
            
        if row['memStatus'] == 'recalled' and not recalledStatusCheck:
            recalledStatusCheck = True
            recalledEuclidDistance = eval(row['euclidDistance'])['recalled']
            
        if imaginedStatusCheck and recalledStatusCheck:
            if imaginedEuclidDistance < recalledEuclidDistance:
                imaginedStoryLowerDistance += 1
            elif imaginedEuclidDistance > recalledEuclidDistance:
                recalledStoryLowerDistance += 1
            else:
                print(1)
                
            storyCount += 1
            if storyCount % 50 == 0:
                print(f"{storyCount} Stories Processed")
            
       
            break 

In [ ]:
print(imaginedStoryLowerDistance, recalledStoryLowerDistance) #736 1836